In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn

In [2]:

context_length = 8
batch_size = 4
embedding_dim = 10
vocab_size = 40

vocab = torch.arange(0, vocab_size)

dataset = torch.zeros([4, 8], dtype=float)

for i in range(len(dataset)):
    indices = torch.randint(0, vocab_size, (context_length,))
    dataset[i] = vocab[indices]

#print(dataset)

# embeddings = torch.rand_like(torch.zeros([vocab_size, embedding_dim]))
embeddings = nn.Embedding(vocab_size, embedding_dim)


# token_embeddings = dataset * embeddings



In [3]:

# token_embeddings  = torch.zeros([batch_size, context_length, embedding_dim], dtype=float)

# for i, data in enumerate(dataset):
#     token_embeddings[i] = embeddings[data.int()]

token_embeddings = embeddings(dataset.int())



In [4]:
Q = nn.Linear(embedding_dim, embedding_dim)
K = nn.Linear(embedding_dim, embedding_dim)
V = nn.Linear(embedding_dim, embedding_dim)

In [5]:
x = embeddings(dataset.int())

x = Q(x)
x = K(x)
x = V(x)


In [6]:
print(f"Data Matrix: {dataset.shape}")
print(f"Embeddings matrix: {embeddings.weight.shape}")
print(f"Token Embeddings: {token_embeddings.shape}")

print("")

print(f"Size of Q: {Q.weight.size()}")
print(f"Size of K: {K.weight.size()}")
print(f"Size of V: {V.weight.size()}")

print("")
print(f"Size of Q(x): {Q(x).size()}")
print(f"Size of K(x): {K(x).size()}")
print(f"Size of V(x): {V(x).size()}")

Data Matrix: torch.Size([4, 8])
Embeddings matrix: torch.Size([40, 10])
Token Embeddings: torch.Size([4, 8, 10])

Size of Q: torch.Size([10, 10])
Size of K: torch.Size([10, 10])
Size of V: torch.Size([10, 10])

Size of Q(x): torch.Size([4, 8, 10])
Size of K(x): torch.Size([4, 8, 10])
Size of V(x): torch.Size([4, 8, 10])


In [17]:
x = embeddings(dataset.int())

k = K(x)

q = Q(x)
v = V(x)

d = torch.tensor(len(embeddings.weight))

print(q.shape)
print(k.shape)

qk = q@k.transpose(-2, -1)

print(f"qk {qk.shape}")

qk_scaled = qk / torch.sqrt(torch.tensor(embedding_dim))

pastmask = torch.tril(torch.ones(batch_size, context_length, context_length))
qk_scaled[pastmask==0] = -torch.inf

qk_softmax = F.softmax(qk_scaled, dim=-1)

actsManual = qk_softmax @ v

print(f"Shape of activations (manual): {actsManual.shape}")

# A = torch.softmax((qk) / torch.sqrt(torch.tensor(embedding_dim)), -1) * v

# print(A.shape)
# print(qk.shape)



torch.Size([4, 8, 10])
torch.Size([4, 8, 10])
qk torch.Size([4, 8, 8])
Shape of activations (manual): torch.Size([4, 8, 10])


In [18]:
actsTorch = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f"Shape of activations (PyTorch): {actsTorch.shape}")

Shape of activations (PyTorch): torch.Size([4, 8, 10])
